# Repository-based Datasources: End-to-End Tutorial

This notebook demonstrates how to work with **repository-aware** datasources in Hera.

Starting with this version, **every datasource is identified by: (repository name, datasource name)**.
You can also set a **project-level default repository** to keep backward compatibility when retrieving by name only.

## What you'll learn
1. Proper environment setup (ensuring your venv and code paths are used).
2. How to **register** a datasource with a repository using the CLI.
3. How to set a **default repository** for a project.
4. How to **retrieve** a datasource by *(repository, name)* or by name only (via default repository).
5. How to instantiate the underlying Python class using the stored classpath + params.
6. Optional usage calls and a short troubleshooting guide.

## Prerequisites
- A working Hera virtual environment (venv) activated.
- Your external toolkit code (e.g., IMS) available locally (e.g. `~/hera-ims/code`).
- If your class relies on additional modules (e.g., `tqdm`, `requests`, `pandas`, `dask`,
  `fastparquet`, `pyarrow`, `python-dateutil`), please ensure they are installed in your Hera venv.


## 1) Environment Setup
We make sure the Hera venv site-packages is first in `sys.path`, followed by the external code paths
(e.g., IMS code folder and its root), and any other dependencies (e.g., `pyargos`).
This helps avoid accidental imports from user-site or system packages (like ParaView).

In [ ]:
import os, sys, pathlib

PROJECT   = os.environ.get("PROJECT", "UnitTestProject")
IMS_ROOT  = os.path.expanduser(os.environ.get("IMS_ROOT" , "~/hera-ims"))
IMS_CODE  = os.path.join(IMS_ROOT, "code")
IMS_DATA  = os.path.join(IMS_ROOT, "data")
PYARGOS   = os.path.expanduser(os.environ.get("PYARGOS_ROOT", "~/pyargos-master"))

# Ensure folders exist (no-op if already there)
pathlib.Path(IMS_CODE).mkdir(parents=True, exist_ok=True)
pathlib.Path(IMS_DATA).mkdir(parents=True, exist_ok=True)
# Make 'code' importable as a top-level package if you wish to use code.XXX imports
open(os.path.join(IMS_CODE, "__init__.py"), "a").close()

# venv site-packages (adjust Python version if different)
VENV_SP = os.path.expanduser("~/hera/heraenv/lib/python3.9/site-packages")

# Make sure user-site is disabled (prevents conflicts with system/user packages)
os.environ["PYTHONNOUSERSITE"] = "1"

for p in (VENV_SP, IMS_CODE, IMS_ROOT, PYARGOS):
    if p and p not in sys.path:
        sys.path.insert(0, p)

print("✔ Environment ready")
print("  Project   :", PROJECT)
print("  IMS_ROOT  :", IMS_ROOT)
print("  IMS_CODE  :", IMS_CODE)
print("  PYARGOS   :", PYARGOS)
print("  sys.path[0:5] =", sys.path[0:5])


## 2) Register a Datasource with a Repository (CLI)
Use the Hera CLI to register your toolkit class as a **ToolkitDataSource**, stored under a given repository.

Key flags:
- `--project`: Hera project name.
- `--repository`: the repository umbrella (logical grouping) for datasources.
- `--name`: the datasource identifier **within** that repository.
- `--classpath`: fully-qualified class (e.g., `IMS_experiment.IMS_experiment` or `code.IMS_experiment.IMS_experiment`).
- `--resource`: optional **folder** to add to `sys.path` before import (useful for bare modules).
- `--params`: JSON of constructor kwargs.

> Tip: If your class imports a sibling module like `presentation`, ensure both the toolkit folder (`IMS_CODE`) **and** its parent root are on `PYTHONPATH`.

In [ ]:
import os, sys, json, subprocess

params = json.dumps({
    "projectName": os.environ.get("PROJECT", "UnitTestProject"),
    "pathToExperiment": os.path.expanduser(os.environ.get("IMS_ROOT", "~/hera-ims")),
    "filesDirectory": os.path.expanduser(os.environ.get("IMS_DATA", "~/hera-ims/data")),
})

env = os.environ.copy()
# Make venv + IMS + IMS root + pyargos importable for the CLI process
env["PYTHONNOUSERSITE"] = "1"
env["PYTHONPATH"] = os.pathsep.join([
    os.path.expanduser("~/hera/heraenv/lib/python3.9/site-packages"),
    os.path.expanduser("~/hera-ims/code"),
    os.path.expanduser("~/hera-ims"),
    os.path.expanduser("~/pyargos-master"),
    env.get("PYTHONPATH","")
])

# Choose a classpath that matches your layout:
# - If IMS_experiment.py is a top-level module inside IMS_CODE:
classpath = "IMS_experiment.IMS_experiment"
# - If you want to import as package 'code.IMS_experiment', you can use:
# classpath = "code.IMS_experiment.IMS_experiment"

print("Registering datasource...")
subprocess.run([
    sys.executable, "-m", "hera.utils.data.cli_toolkit_repository", "register-datasource",
    "--project",  os.environ.get("PROJECT", "UnitTestProject"),
    "--repository", "IMS",
    "--name", "IMS",
    "--classpath", classpath,
    "--resource", os.path.expanduser("~/hera-ims/code"),
    "--params", params,
    "--version", "0.0.1",
    "--overwrite",
], check=True, env=env)

print("✔ Registered 'IMS' datasource in repository 'IMS'")


## 3) Set the Project's Default Repository
Setting a **default repository** allows you to retrieve a datasource by **name only** (backward compatible),
without passing `repository` on every call.

In [ ]:
import os, sys, subprocess

print("Setting default repository for the project...")
subprocess.run([
    sys.executable, "-m", "hera.utils.data.cli_toolkit_repository", "set-default-repository",
    "--project",  os.environ.get("PROJECT", "UnitTestProject"),
    "--repository", "IMS",
], check=True)
print("✔ Default repository set to 'IMS'")


## 4) Retrieve the Datasource (with/without repository)
We show both:
- Retrieval by **name only** (falls back to the default repository we just set).
- Retrieval by **explicit (repository, name)** (which overrides default).

In [ ]:
import os, sys, subprocess

print("\n-- implicit default repository --")
subprocess.run([
    sys.executable, "-m", "hera.utils.data.cli_toolkit_repository", "get-datasource",
    "--project",  os.environ.get("PROJECT", "UnitTestProject"),
    "--name", "IMS",
], check=True)

print("\n-- explicit repository --")
subprocess.run([
    sys.executable, "-m", "hera.utils.data.cli_toolkit_repository", "get-datasource",
    "--project",  os.environ.get("PROJECT", "UnitTestProject"),
    "--repository", "IMS",
    "--name", "IMS",
    "--version", "0.0.1",
], check=True)


## 5) Project API: Fetch Document & Instantiate the Class
Below we fetch the **ToolkitDataSource** document from the DB using the Project API
and instantiate the underlying class using the stored `classpath` and `parameters`.
This confirms that your `PYTHONPATH` and dependencies are consistent.

In [ ]:
import os, sys, importlib
from hera.datalayer.project import Project

# Keep the critical import paths at the front
for p in (os.path.expanduser("~/hera/heraenv/lib/python3.9/site-packages"),
          os.path.expanduser("~/hera-ims/code"),
          os.path.expanduser("~/hera-ims"),
          os.path.expanduser("~/pyargos-master")):
    if p not in sys.path:
        sys.path.insert(0, p)

proj = Project(projectName=os.environ.get("PROJECT", "UnitTestProject"))
docs = proj.getMeasurementsDocuments(type="ToolkitDataSource",
                                     repository="IMS",
                                     datasourceName="IMS")
assert docs, "Could not find datasource 'IMS' in repository 'IMS'. Did you register it above?"
doc = docs[0]
print("Document ✅")
print("  repository:", doc.desc.get("repository"))
print("  name      :", doc.desc.get("datasourceName"))
print("  classpath :", doc.desc.get("classpath"))
print("  version   :", doc.desc.get("version"))
print("  resource  :", doc.resource)

module_name, _, class_name = doc.desc["classpath"].rpartition(".")
Cls = getattr(importlib.import_module(module_name), class_name)
params = doc.desc["parameters"]
obj = Cls(**params)
print("Instance ✅ ->", type(obj))


## 6) Optional Usage Examples
The following calls are **commented out**. Enable them only if you have a valid IMS token at `~/hera-ims/token.json`.
They may take time and write parquet data under `IMS/data/`.


In [ ]:
# station_name = "YAVNEEL"   # choose a station that exists in your setup
# obj.download(station=station_name, start="2020-01-01", end="latest")
# obj.update(station=station_name, end="latest")
print("Ready. Uncomment usage once token & paths are configured.")


## 7) Troubleshooting
- **`ModuleNotFoundError: presentation`** — make sure both `IMS/code` **and** `IMS` root are on `sys.path`.
- **Missing packages (e.g. `tqdm`, `requests`, `pandas`)** — install them into the Hera venv.
- **No default repository set** — call `set-default-repository` first or pass `--repository` to calls.
- **Wrong import source** — ensure `PYTHONNOUSERSITE=1` to avoid loading from user-site/system packages.
